# 1. TITLE
## Design of a Simple Web Crawler with Depth and Domain Restrictions

---

## 2. AIM
The aim of this practical is to design, implement, and analyze a simple, beginner-friendly web crawler in Python.

Specifically, the crawler will incorporate:
- **Depth restriction** to prevent the crawler from running infinitely.
- **Domain restriction** to ensure the crawler only visits pages belonging to a specific target domain.
- **Visited URL tracking** to avoid crawling the same webpage multiple times.
- **Hyperlink extraction** to discover new URLs on crawled pages.
- **Page metadata storage** to record useful information about each crawled page for analysis.

## 3. THEORY

- **What is a web crawler?**
  A web crawler (or spider) is an automated program that systematically browses the World Wide Web, typically for the purpose of web indexing.

- **What is a seed URL?**
  The seed URL is the starting point of the crawl. The crawler begins downloading and extracting links from this initial page.

- **What is crawling depth?**
  Crawling depth refers to how many links away a page is from the seed URL. The seed URL is at depth 0. Pages linked directly from the seed are at depth 1, and pages linked from depth 1 are at depth 2.

- **What is domain restriction?**
  Domain restriction limits the crawler to only visit links belonging to a specific target host (e.g., `quotes.toscrape.com`), preventing it from wandering off to external websites.

- **Why do we maintain a visited set?**
  Websites often have cyclic links (Page A links to Page B, and Page B links to Page A). A visited set tracks already crawled URLs to avoid infinite loops and redundant network requests.

- **What is a queue?**
  A queue is a First-In, First-Out (FIFO) data structure. We use it to store URLs that are discovered but not yet crawled.

- **Why is BFS suitable for a simple crawler?**
  Breadth-First Search (BFS) is implemented using a queue. It explores all pages at the current depth before moving deeper, making it perfect for enforcing a maximum depth restriction.

- **What metadata can be stored for each crawled page?**
  Useful metadata includes the page URL, crawl depth, HTTP status code, page HTML title, and the number of links found on that page.

## 4. ALGORITHM

1. **Step 1:** Start with a seed URL.
2. **Step 2:** Put the seed URL in a queue with depth 0.
3. **Step 3:** Remove a URL from the queue.
4. **Step 4:** Check whether it has already been visited. If yes, skip to Step 3.
5. **Step 5:** Check whether its depth is within the maximum depth. If it exceeds, skip.
6. **Step 6:** Check whether the URL belongs to the allowed domain. If not, skip.
7. **Step 7:** Download the webpage content.
8. **Step 8:** Parse the HTML using BeautifulSoup.
9. **Step 9:** Extract the page title and hyperlinks.
10. **Step 10:** Store page metadata (URL, depth, status code, title, link count).
11. **Step 11:** Add valid links to the queue with depth + 1.
12. **Step 12:** Continue until the queue is empty.
13. **Step 13:** Display the crawler metadata using pandas.

## 5. IMPORT LIBRARIES
We will now import the standard library components and packages needed for fetching webpages, parsing HTML structure, and presenting data.

In [1]:
# Design a simple web Crawler with depth and domain restrictions. Store crawler pages metadata

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urlparse, urljoin
from collections import deque

## 6. IMPLEMENTATION
Here is the implementation of our simple crawler. It uses a FIFO queue (`collections.deque`) and a tracking set (`visited`) to process and store metadata.

In [2]:
def crawl(seed_url, max_depth=2, allowed_domain=None):
    # Initialize BFS queue with (url, current_depth)
    queue = deque([(seed_url, 0)])
    # Set to keep track of visited URLs
    visited = set()
    # List to store metadata dictionaries
    metadata_list = []

    print(f"Starting crawl on: {seed_url} (Max Depth: {max_depth}, Allowed Domain: {allowed_domain})\n")

    while queue:
        url, depth = queue.popleft()

        # Skip if already visited
        if url in visited:
            continue

        # Skip if depth exceeds limit
        if depth > max_depth:
            continue

        # Parse domain to enforce restriction
        parsed_url = urlparse(url)
        if allowed_domain and parsed_url.netloc != allowed_domain:
            continue

        # Mark as visited
        visited.add(url)

        print(f"Crawling: {url} at depth {depth}")

        try:
            # Fetch webpage with a timeout of 5 seconds
            response = requests.get(url, timeout=5)
            status_code = response.status_code

            if status_code == 200:
                soup = BeautifulSoup(response.text, "html.parser")

                # Extract title
                title = soup.title.string.strip() if soup.title and soup.title.string else "No Title"

                # Extract links
                links_found = []
                for link_tag in soup.find_all('a', href=True):
                    href = link_tag['href']
                    # Resolve relative URLs to absolute URLs
                    full_url = urljoin(url, href)
                    links_found.append(full_url)

                # Add metadata
                metadata_list.append({
                    "URL": url,
                    "Depth": depth,
                    "Status Code": status_code,
                    "Title": title,
                    "Number of Links": len(links_found)
                })

                # Add newly found links to queue with depth + 1
                for next_link in links_found:
                    if next_link not in visited:
                        queue.append((next_link, depth + 1))
            else:
                # Record unsuccessful attempts gracefully
                metadata_list.append({
                    "URL": url,
                    "Depth": depth,
                    "Status Code": status_code,
                    "Title": "Failed to Load",
                    "Number of Links": 0
                })

        except Exception as e:
            # Handle exceptions (e.g., DNS lookup failures, connection timeouts)
            print(f"  Error crawling {url}: {e}")
            metadata_list.append({
                    "URL": url,
                    "Depth": depth,
                    "Status Code": "Error",
                    "Title": str(e)[:30],
                    "Number of Links": 0
            })

    return metadata_list

## 7. RUN THE CRAWLER
We will crawl the safe playground website `https://quotes.toscrape.com/` with a maximum depth of 2 and restrict the domain to `quotes.toscrape.com`.

In [3]:
seed_url = "https://quotes.toscrape.com/"
max_depth = 2
allowed_domain = "quotes.toscrape.com"

# Run the crawler
crawled_data = crawl(seed_url, max_depth, allowed_domain)

# Convert to pandas DataFrame
df = pd.DataFrame(crawled_data)

# Display DataFrame
display(df)

Starting crawl on: https://quotes.toscrape.com/ (Max Depth: 2, Allowed Domain: quotes.toscrape.com)

Crawling: https://quotes.toscrape.com/ at depth 0
Crawling: https://quotes.toscrape.com/login at depth 1
Crawling: https://quotes.toscrape.com/author/Albert-Einstein at depth 1
Crawling: https://quotes.toscrape.com/tag/change/page/1/ at depth 1
Crawling: https://quotes.toscrape.com/tag/deep-thoughts/page/1/ at depth 1
Crawling: https://quotes.toscrape.com/tag/thinking/page/1/ at depth 1
Crawling: https://quotes.toscrape.com/tag/world/page/1/ at depth 1
Crawling: https://quotes.toscrape.com/author/J-K-Rowling at depth 1
Crawling: https://quotes.toscrape.com/tag/abilities/page/1/ at depth 1
Crawling: https://quotes.toscrape.com/tag/choices/page/1/ at depth 1
Crawling: https://quotes.toscrape.com/tag/inspirational/page/1/ at depth 1
Crawling: https://quotes.toscrape.com/tag/life/page/1/ at depth 1
Crawling: https://quotes.toscrape.com/tag/live/page/1/ at depth 1
Crawling: https://quotes.to

,URL,Depth,Status Code,Title,Number of Links
0,https://quotes.toscrape.com/,0,200,Quotes to Scrape,55
1,https://quotes.toscrape.com/login,1,200,Quotes to Scrape,4
2,https://quotes.toscrape.com/author/Albert-Eins...,1,200,Quotes to Scrape,4
3,https://quotes.toscrape.com/tag/change/page/1/,1,200,Quotes to Scrape,20
4,https://quotes.toscrape.com/tag/deep-thoughts/...,1,200,Quotes to Scrape,20
...,...,...,...,...,...
144,https://quotes.toscrape.com/tag/read/page/1/,2,200,Quotes to Scrape,20
145,https://quotes.toscrape.com/tag/readers/page/1/,2,200,Quotes to Scrape,20
146,https://quotes.toscrape.com/tag/reading-books/...,2,200,Quotes to Scrape,20
147,https://quotes.toscrape.com/author/Alfred-Tenn...,2,200,Quotes to Scrape,4


## 8. DEMONSTRATE DEPTH RESTRICTION
Let's count how many pages were successfully crawled at each depth. Depth 0 represents the starting seed page, depth 1 contains pages linked directly from the seed page, and depth 2 contains pages reached from depth 1.

In [4]:
depth_counts = df.groupby('Depth').size().reset_index(name='Page Count')
display(depth_counts)

,Depth,Page Count
0,0,1
1,1,46
2,2,102


## 9. DEMONSTRATE DOMAIN RESTRICTION
During parsing, any links pointing outside of `quotes.toscrape.com` (such as links to external social media sites) were automatically discarded. Let's verify that all crawled URLs indeed belong to the permitted domain.

In [5]:
# Extract domains from crawled URLs
domains_found = df['URL'].apply(lambda u: urlparse(u).netloc).unique()
print("Domains crawled:", domains_found)

Domains crawled: ['quotes.toscrape.com']


## 10. DISPLAY METADATA
Here is the consolidated preview of our crawled data and summary metrics.

In [6]:
print(f"Total pages crawled: {len(df)}")
display(df[["URL", "Depth", "Status Code", "Title", "Number of Links"]])

Total pages crawled: 149


,URL,Depth,Status Code,Title,Number of Links
0,https://quotes.toscrape.com/,0,200,Quotes to Scrape,55
1,https://quotes.toscrape.com/login,1,200,Quotes to Scrape,4
2,https://quotes.toscrape.com/author/Albert-Eins...,1,200,Quotes to Scrape,4
3,https://quotes.toscrape.com/tag/change/page/1/,1,200,Quotes to Scrape,20
4,https://quotes.toscrape.com/tag/deep-thoughts/...,1,200,Quotes to Scrape,20
...,...,...,...,...,...
144,https://quotes.toscrape.com/tag/read/page/1/,2,200,Quotes to Scrape,20
145,https://quotes.toscrape.com/tag/readers/page/1/,2,200,Quotes to Scrape,20
146,https://quotes.toscrape.com/tag/reading-books/...,2,200,Quotes to Scrape,20
147,https://quotes.toscrape.com/author/Alfred-Tenn...,2,200,Quotes to Scrape,4


## 11. WORKING EXPLANATION

- **`queue = deque()`**:
  Initializes a double-ended queue. We use it as a First-In, First-Out (FIFO) structure to perform Breadth-First Search (BFS) crawling.

- **`visited = set()`**:
  A Python set utilized to keep track of unique URLs we have already visited, ensuring we do not run into an infinite crawl loop.

- **`queue.append((url, depth))`**:
  Pushes a new tuple containing the URL and its distance (depth) from the seed into the queue.

- **`urlparse(url).netloc`**:
  Extracts the domain portion of a URL (for example, parsing `https://quotes.toscrape.com/page/1` yields `quotes.toscrape.com`), allowing us to enforce domain restrictions.

- **`urljoin(url, link)`**:
  Combines a base URL with a relative path (e.g., `/login`) to construct a valid absolute URL (e.g., `https://quotes.toscrape.com/login`).

- **`depth + 1`**:
  Increments the current parent webpage's depth by 1 and assigns it to all newly discovered child URLs so we do not cross our depth limit.

- **`BeautifulSoup(response.text, "html.parser")`**:
  Parses raw HTML text from our HTTP response into a structured tree object, making it easy to search for elements like `<title>` and hyperlink anchor tags `<a>`.

## 12. SAMPLE FLOW
```
Seed URL (Depth 0)
      │
      ▼
  Fetch page ────► Handle Exception / Store Status Error
      │
      ▼ (Success)
Extract HTML Links
      │
      ▼
Check Domain (Is it 'quotes.toscrape.com'?)
      │
      ▼ (Yes)
Check Depth (Is current Depth <= 2?)
      │
      ▼ (Yes)
Add valid link to Queue (with Depth + 1)
      │
      ▼
Store Metadata (URL, Depth, Status, Title, Links Count)
      │
      ▼
Repeat for next link in Queue
```

## 13. VIVA QUESTIONS

1. **What is a web crawler?**
   *Answer:* An automated program that systematically browses the internet to download, parse, and catalog web pages.

2. **What is a seed URL?**
   *Answer:* The initial website address given to the crawler from where the crawling process starts.

3. **What is crawling?**
   *Answer:* The act of visiting webpages, fetching their contents, extracting new hyperlinks, and storing relevant information.

4. **Why do we use a queue?**
   *Answer:* To hold discovered links that need to be visited, keeping track of them in order (FIFO).

5. **Why is BFS useful here?**
   *Answer:* It naturally explores layer by layer (level by level), which makes limiting the maximum depth extremely straightforward.

6. **What is depth restriction?**
   *Answer:* A mechanism to stop crawling when a page is too far (too many link clicks away) from the seed page.

7. **What is domain restriction?**
   *Answer:* A constraint that prevents the crawler from visiting URLs belonging to websites outside our target domain.

8. **Why do we use a visited set?**
   *Answer:* To remember which webpages have already been crawled, preventing duplicate network visits and infinite loops.

9. **What is BeautifulSoup used for?**
   *Answer:* It is a library used to parse HTML markup text, making it easy to search for tags, attributes, and text.

10. **What metadata are we storing?**
    *Answer:* We are storing the page URL, crawl Depth, HTTP Status Code, HTML page Title, and the Number of Links found inside it.

## 14. RESULT
The simple web crawler was successfully implemented with depth and domain restrictions. The crawler extracted links from webpages and stored page metadata such as URL, depth, status code, title and number of links.